# BetaPPO Gray84 · 같은 관측에서 평균과 샘플링 비교
기존 Drive 체크포인트로 평가만 실행합니다. 각 관측에서 결정론적 평균 행동과 같은 Beta 분포에서 뽑은 8,192개 행동을 비교합니다. 실제 차량에는 항상 평균 행동만 적용하므로 샘플링 때문에 비교할 관측이 달라지지 않습니다.
기본 트랙은 seed 1000이며 Drive에 있는 GAE와 TD3의 best.pt를 각각 읽습니다. 두 체크포인트의 주행 경로는 서로 달라질 수 있습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys, subprocess, hashlib, zipfile, shutil
from pathlib import Path
from google.colab import drive, files

os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
root = Path('/content/beta_ppo_gray84_distribution_diag')
drive_root = Path('/content/drive/MyDrive/CarRacing')
zip_path = Path('/content/carracing_competition.zip')
if not zip_path.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('공식 ZIP 하나를 선택하세요.')
    zip_path.write_bytes(next(iter(uploaded.values())))
manifest = {'agents/actor_critic.py': 'd25aa847bf5d9c0c9803c0016849f2d9bae410caeecf0d943fba8fd695c02037', 'agents/base.py': 'f183f1e46745a9abb5e7f0fc88a08be134104f599326f57ac9bc7f63a324e743', 'agents/dqn.py': '37d7168f7bcb252eff69aab561e85c5ab3d73f01007f6ab95ab00d60fd0d5415', 'agents/__init__.py': '215cd1e434eaf15791c2c826380c5eab6844225ce7b5b9e134894eefd52edc59', 'buffers.py': '59b576518a9997a51f2811f00c8c900394dba24f45d6fb8eaa05f0324da018f1', 'check_submission.py': '7a62312300656f891d1aeffdcab249c501de0153567138f027e8491998763a02', 'envs.py': 'ceee2d847af1a3a498c6b7f0f83f3561b24ac22e53de8bded43c97a15af645d0', 'evaluate.py': '219d4c3ded5b8624d8e4f69d3c71e27145aed6b9eb20dc0ef8b0f9c2355c2574', 'export.py': 'dbd9fe51c3046d2d5a458ad8ea464e99cde777c13be7d8406bf2f667b05f8a6f', 'networks.py': '05c5c6dbd6f4ce2fbfe2a71ef18f84d175ff7a7ff36540c9172f750f27b2b3d8', 'README.md': 'f4b928b8d1ae7f907d21670f5d5c0fe6050845f056ab567ab7b18968608a86b0', 'requirements.txt': 'b38b63f4f8d53b8edce022020a26886d6f079d3484d16d3eb630e530fd11b9b7', 'train.py': '579b46fbf559233490bd3e0949b29880dfe564d890e15c37ba99db7a2344b6b1'}
with zipfile.ZipFile(zip_path) as archive:
    for name, digest in manifest.items():
        data = archive.read('carracing_competition/' + name)
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError(f'공식 배포본 버전을 확인하세요: {name}')
        dest = root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# Resample the full field of view with the same pixel indices in NumPy and ONNX.
env_file = root/'envs.py'
original_env = env_file.read_text()
edits = [
    ('N_STACK = 4', 'GRAY84_INDEX = (np.arange(84) * 96 + 48) // 84\nN_STACK = 4'),
    ('        h, w, _ = env.observation_space.shape\n        self.observation_space = gym.spaces.Box(0, 255, (h, w), np.uint8)',
     '        self.observation_space = gym.spaces.Box(0, 255, (84, 84), np.uint8)'),
    ('return gray.astype(np.uint8)', 'return gray[GRAY84_INDEX[:, None], GRAY84_INDEX].astype(np.uint8)'),
    ('return gray.floor().to(torch.uint8)', 'idx = torch.as_tensor(GRAY84_INDEX, device=gray.device)\n    return gray.index_select(-2, idx).index_select(-1, idx).floor().to(torch.uint8)'),
    ('-> (96, 96) uint8 흑백', '-> (84, 84) uint8 흑백'),
    ('여기서는 (B, N_STACK, 96, 96) uint8', '여기서는 (B, N_STACK, 84, 84) uint8'),
]
for old, new in edits:
    assert original_env.count(old) == 1, old
    original_env = original_env.replace(old, new)
env_file.write_text(original_env)


In [ ]:
%%writefile agents/beta_ppo.py
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import Beta
from agents.base import Agent
from networks import CNNEncoder


def controls(sample):
    steer, pedal = (2 * sample - 1).unbind(-1)
    return torch.stack((steer, pedal.relu(), (-pedal).relu()), dim=-1)


class BetaPolicy(nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.encoder = CNNEncoder(shape, feature_dim=256)
        self.actor = nn.Linear(256, 4)
        self.progress = nn.Linear(258, 1)
        self.critic = nn.Linear(256, 1)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        # Mild forward exploration at initialization. All four biases are trainable.
        with torch.no_grad():
            self.actor.bias.copy_(torch.tensor([0., 1.5, 0., -0.5]))

    def heads(self, features):
        alpha, beta = (1 + F.softplus(self.actor(features))).chunk(2, dim=-1)
        return alpha, beta, self.critic(features).squeeze(-1)

    def parameters_at(self, obs):
        return self.heads(self.encoder(obs))

    def forward(self, obs):
        alpha, beta, _ = self.parameters_at(obs)
        return controls(alpha / (alpha + beta))


def rollout_advantages(rewards, values, next_values, terms, truncs, config):
    """Compute GAE or a fixed-horizon TD advantage without crossing lane boundaries."""
    gamma = config["gamma"]
    horizon = config["td_steps"]
    delta = (np.asarray(rewards) * config["reward_scale"] - np.asarray(values)
             + gamma * np.where(terms, 0., next_values))
    boundary = np.logical_or(terms, truncs)
    advantages = np.empty(len(delta), dtype=np.float32)
    if horizon == 0:
        tail = 0.
        for i in reversed(range(len(delta))):
            carry = 0. if boundary[i] else tail
            tail = delta[i] + gamma * config["gae_lambda"] * carry
            advantages[i] = tail
        return advantages

    for i in range(len(delta)):
        total = 0.
        for j in range(i, min(i + horizon, len(delta))):
            total += gamma ** (j - i) * delta[j]
            if boundary[j]:
                break
        advantages[i] = total
    return advantages


class BetaPPO(Agent):
    name = "beta_ppo"
    default_config = dict(continuous=True, lr=0.00025, gamma=0.99, gae_lambda=0.95,
                          td_steps=0, rollout=512, epochs=4, batch=64, clip=0.2, reward_scale=0.1,
                          progress_coef=0.05, entropy_coef=0.005, target_kl=0.025)

    def __init__(self, obs_shape, action_space, config, device):
        super().__init__(obs_shape, action_space, config, device)
        if not self.continuous:
            raise ValueError("beta_ppo requires continuous=True")
        self.net = BetaPolicy(obs_shape).to(device)
        self.optim = torch.optim.Adam(self.net.parameters(), lr=self.cfg["lr"], eps=1e-5)
        self.buffer = []

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        x = torch.as_tensor(obs, device=self.device)[None]
        if deterministic:
            return self.net(x)[0].cpu().numpy()
        alpha, beta, value = self.net.parameters_at(x)
        dist = Beta(alpha, beta)
        sample = dist.sample()
        self.pending = (sample[0].cpu().numpy(), dist.log_prob(sample).sum().item(), value.item())
        return controls(sample)[0].cpu().numpy()

    def step(self, obs, action, reward, next_obs, terminated, truncated):
        self.buffer.append((obs, *self.pending, reward, next_obs, terminated, truncated))
        if len(self.buffer) < self.cfg["rollout"]:
            return {}
        metrics = self.update()
        self.buffer.clear()
        return metrics

    def update(self):
        c = self.cfg
        obs, samples, logs, values, rewards, successors, terms, truncs = zip(*self.buffer)
        def tensor(seq, dtype):
            return torch.as_tensor(np.stack(seq), device=self.device, dtype=dtype)
        x = tensor(obs, torch.uint8)
        u = tensor(samples, torch.float32)
        old_log = tensor(logs, torch.float32)
        with torch.no_grad():
            next_x = tensor(successors, torch.uint8)
            next_values = torch.cat([self.net.parameters_at(chunk)[2] for chunk in next_x.split(c["batch"])]).cpu().numpy()
        advantages = rollout_advantages(rewards, values, next_values, terms, truncs, c)
        adv = tensor(advantages, torch.float32)
        targets = adv + tensor(values, torch.float32)
        adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)
        gain = tensor(np.asarray(rewards) > 0, torch.float32)
        losses = []
        for epoch in range(c["epochs"]):
            divergences = []
            for idx in torch.randperm(len(x), device=self.device).split(c["batch"]):
                features = self.net.encoder(x[idx])
                a, b, pred = self.net.heads(features)
                dist = Beta(a, b)
                log_ratio = dist.log_prob(u[idx]).sum(-1) - old_log[idx]
                ratio = log_ratio.exp()
                clipped = ratio.clamp(1-c["clip"], 1+c["clip"])
                policy_loss = -torch.minimum(ratio * adv[idx], clipped * adv[idx]).mean()
                loss = policy_loss + 0.5 * F.mse_loss(pred, targets[idx])
                progress_logits = self.net.progress(torch.cat((features, u[idx]), dim=1)).squeeze(-1)
                loss += c["progress_coef"] * F.binary_cross_entropy_with_logits(progress_logits, gain[idx])
                loss -= c["entropy_coef"] * dist.entropy().sum(-1).mean()
                self.optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5, error_if_nonfinite=True)
                self.optim.step()
                losses.append(loss.item())
                divergences.append((ratio.detach() - 1 - log_ratio.detach()).mean().item())
            if np.mean(divergences) > c["target_kl"]:
                break
        return dict(loss=float(np.mean(losses)), kl=float(np.mean(divergences)),
                    gas=float((2*u[:,1]-1).relu().mean()), brake=float((1-2*u[:,1]).relu().mean()))

    def export_policy(self):
        return self.net

    def state_dict(self):
        return dict(net=self.net.state_dict(), optim=self.optim.state_dict())

    def load_state_dict(self, state):
        self.net.load_state_dict(state["net"])
        self.optim.load_state_dict(state["optim"])
        self.buffer.clear()  # A resumed environment starts a new episode.

In [ ]:
from collections import deque
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
from envs import preprocess_torch, FRAME_SKIP, N_STACK
from evaluate import load_agent
from check_submission import HideScoreText

SEED = 1000
VARIANTS = ('gae', 'td3')
CHECKPOINT = 'best.pt'
N_SAMPLES = 8192
ACTION_NAMES = ('steer', 'gas', 'brake')
torch.set_num_threads(1)

def distribution_summary(alpha, beta, rng):
    mean_latent = alpha / (alpha + beta)
    steer, pedal = 2 * mean_latent - 1
    mean_action = np.array([steer, max(pedal, 0), max(-pedal, 0)])
    sampled_steer, sampled_pedal = (2 * rng.beta(alpha, beta, size=(N_SAMPLES, 2)) - 1).T
    samples = np.stack((sampled_steer, sampled_pedal.clip(0), (-sampled_pedal).clip(0)), axis=1)
    lo, hi = np.quantile(samples, [.05, .95], axis=0)
    result = dict(p_brake=float(np.mean(sampled_pedal < 0)),
                  p_brake_ge_010=float(np.mean(sampled_pedal <= -.10)),
                  p_steer_left_ge_025=float(np.mean(sampled_steer <= -.25)),
                  p_steer_right_ge_025=float(np.mean(sampled_steer >= .25)))
    for i, name in enumerate(ACTION_NAMES):
        result.update({f'eval_{name}': mean_action[i], f'sampled_mean_{name}': samples[:, i].mean(),
                       f'{name}_p05': lo[i], f'{name}_p95': hi[i],
                       f'{name}_mae': np.abs(samples[:, i]-mean_action[i]).mean()})
    return result

def grass_fraction(rgb):
    strips = np.concatenate((rgb[66:76,39:43], rgb[66:76,53:57]), axis=1).astype(int)
    red, green, blue = strips.transpose(2,0,1)
    return ((green > red + 25) & (green > blue + 25)).mean()

@torch.no_grad()
def inspect_checkpoint(checkpoint):
    agent = load_agent(str(checkpoint), torch.device('cpu'))
    agent.net.eval()
    rng = np.random.default_rng(SEED)
    env = HideScoreText(gym.make('CarRacing-v3', continuous=True))
    records = []
    hints = deque(maxlen=5)
    frame = 0
    entry = None
    done = False
    try:
        obs, _ = env.reset(seed=SEED)
        stack = deque([obs] * N_STACK, maxlen=N_STACK)
        while not done:
            batch = preprocess_torch(torch.from_numpy(np.stack(stack)[None]))
            alpha, beta, _ = agent.net.parameters_at(batch)
            alpha, beta = alpha[0].numpy(), beta[0].numpy()
            stats = distribution_summary(alpha, beta, rng)
            action = agent.act(batch[0].numpy(), deterministic=True)
            np.testing.assert_allclose(action, [stats[f'eval_{name}'] for name in ACTION_NAMES], atol=1e-6)
            records.append(dict(first_action_frame=frame+1, observed_frame=frame,
                                steer_alpha=alpha[0], steer_beta=beta[0],
                                pedal_alpha=alpha[1], pedal_beta=beta[1], **stats))
            for _ in range(FRAME_SKIP):
                obs, _, terminated, truncated, _ = env.step(action)
                frame += 1
                hints.append(grass_fraction(obs))
                if entry is None and frame >= 60 and len(hints)==5 and min(hints) > .65:
                    entry = frame-4
                done = terminated or truncated
                if done:
                    break
            stack.append(obs)
    finally:
        env.close()
    trace = pd.DataFrame(records)
    trace['first_grass_frame'] = entry
    if entry is not None:
        trace['relative_frame'] = trace.first_action_frame - entry
    return trace, entry


In [ ]:
for variant in VARIANTS:
    run = drive_root / f'BetaPPO_Gray84_s42_{variant}'
    checkpoint = run / CHECKPOINT
    if not checkpoint.is_file():
        print(f'{variant}: 체크포인트 없음 ({checkpoint})')
        continue
    trace, entry = inspect_checkpoint(checkpoint)
    out = run / 'diagnostics' / 'action_distribution'
    out.mkdir(parents=True, exist_ok=True)
    stem = out / f'seed_{SEED}_{checkpoint.stem}'
    trace.to_csv(stem.with_suffix('.csv'), index=False)
    if entry is None:
        window = trace
        print(f'{variant}: 잔디 이탈이 감지되지 않아 전체 주행을 표시합니다.')
    else:
        window = trace[trace.relative_frame.between(-50, 40)]
        print(f'{variant}: 첫 이탈 추정 {entry}프레임. 아래 표는 이탈 직전 50프레임의 정책 결정입니다.')
    before = window if entry is None else window[window.relative_frame.le(0)]
    columns = ['first_action_frame', 'eval_steer', 'steer_p05', 'steer_p95',
               'eval_gas', 'eval_brake', 'p_brake', 'p_brake_ge_010', 'steer_mae']
    display(before[columns].round(3))
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(9, 7), layout='constrained')
    for ax, name in zip(axes, ACTION_NAMES):
        x = window.first_action_frame.to_numpy()
        ax.fill_between(x, window[f'{name}_p05'].to_numpy(), window[f'{name}_p95'].to_numpy(),
                        alpha=.2, label='Sample 5–95%')
        ax.plot(x, window[f'eval_{name}'], label='Deterministic action')
        ax.plot(x, window[f'sampled_mean_{name}'], '--', label='Sample mean')
        if entry is not None:
            ax.axvline(entry, color='gray', linestyle=':')
        ax.set_ylabel(name)
    axes[0].legend()
    axes[-1].set_xlabel('First raw frame using this action')
    fig.suptitle(f'{variant} · seed {SEED}')
    fig.savefig(stem.with_suffix('.png'), dpi=150)
    plt.show()
